# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and basic processing of the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj). All data access is performed using the [mlcroissant](https://github.com/mlcommons/croissant) library, referencing schema entities by their `@id` for reproducibility and clarity.

### Dataset Source
Schema: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect its basic properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata is available via dataset.metadata (as an object, not dict/list)
md = dataset.metadata
print(f"{md.name}: {md.description}\n")

print("Dataset fields:\n")
for k in [
    'identifier', 'version', 'datePublished', 'citation', 'conformsTo', 
    'keywords', 'license', 'funding', 'personalSensitiveInformation',
    'dataCollection', 'dataUseCases', 'dataLimitations', 'dataPreprocessingProtocol', 'dataBiases',
    'dataCollectionRawData', 'dataCollectionMissingData', 'dataCollectionType', 'dataCollectionTimeframe'
]:
    if hasattr(md, k):
        print(f"{k}: {getattr(md,k)}")

## 2. Data Overview
The dataset schema may define one or more record sets, each corresponding to a table of related records. We'll enumerate record sets and their fields using their `@id`s.

In [ ]:
# List all record sets using their @id and list fields for each
print("Record sets and their fields (by @id):\n")
record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"- Record Set @id: {rs.id}")
    print(f"    Name: {getattr(rs, 'name', '')}")
    print(f"    Description: {getattr(rs, 'description', '')}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"        - Field @id: {field.id} | name: {getattr(field, 'name', '')} | type: {getattr(field, 'data_type', '')}")
    print()
# Save first record set @id to variable for later use
first_rs_id = record_sets[0].id if record_sets else None

## 3. Data Extraction
We'll load the first available record set (referenced by its `@id`) into a pandas DataFrame. All field/column reference is by their `@id` as listed above.

In [ ]:
# Load data records for each record set (by @id)
# You may specify other record set IDs from above if needed

dataframes = {}
record_sets = dataset.record_sets()
record_set_ids = [rset.id for rset in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @id '{record_set_id}'. Columns:")
    print(df.columns.tolist())
    print(df.head(2), "\n")

# For demonstration, select the first record set for further EDA
rs_id = record_set_ids[0] if record_set_ids else None
df_main = dataframes[rs_id] if rs_id else None

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic data processing on a numeric field, showing referencing by field `@id`:
- Filtering by a threshold
- Normalizing
- Grouping by a categorical field (referenced by `@id` only)

> **Note:** Please review the list of numeric fields in the output above (see columns and field data types) and adjust the variables below accordingly. In the FAIR² dataset, typical numeric fields may include e.g. `gad7_score`, `phq9_score`, or `psq_score` (scores from mental health assessments). Edit the variable values if you want a different field.

In [ ]:
# Example: select a numeric (score) field and a grouping field by their @id (use the printed field list in section 2)
# For this demo, we guess at typical ids; please adjust as needed for your schema!

# List column names for convenience
if df_main is not None:
    print("Available columns in selected DataFrame:", df_main.columns.tolist())

    # --- CHANGE these to match an actual numeric field and group field @id from your listing ---
    # For demonstration, we attempt common fields. Adjust if needed.
    numeric_field_id = 'gad7_score' # e.g. "gad7_score" or "phq9_score" or another numeric field @id
    group_field_id = 'gender'       # e.g. "gender" or "village" or similar group field @id

    if numeric_field_id in df_main.columns:
        # Filter: keep only records with score > threshold
        threshold = 10
        filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)} records):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optional: group by a categorical field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(
                'mean_' + numeric_field_id
            )
            print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
        else:
            print(f"Column '{group_field_id}' not found. Grouping skipped.")
    else:
        print(f"Column '{numeric_field_id}' not found. Please pick an available numeric field.")
else:
    print("No main record set loaded.")

## 5. Visualization
Let's quickly visualize the distribution of the selected numeric field (e.g. assessment score) and optionally scores by group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if df_main is not None and numeric_field_id in df_main.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df_main[numeric_field_id], bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    if group_field_id in df_main.columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print(f"Column '{numeric_field_id}' not found in data. Visualization skipped.")

## 6. Conclusion
We have demonstrated basic use of the `mlcroissant` library to:
- Load a Croissant dataset by schema URL
- Explore its record sets, fields, and data by `@id`
- Load records into pandas DataFrames using only the schema `@id` for referencing
- Apply simple data processing, filtering, normalization, and grouping
- Visualize distributions and group trends

For advanced analysis, refer further to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the dataset [metadata](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).